## 07-ENTRENAMIENTO MODELO RNN-LSTM ##

In [97]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, Callback
import matplotlib.pyplot as plt
import os, logging
from tabulate import tabulate

In [98]:
# Logging mínimo
logging.basicConfig(
    filename="../LOGS/07-entrenamiento_lstm.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [99]:
logging.info("1. CONFIGURACIÓN Y CARGA DE DATOS GLOBAL")

# --- 1. CONFIGURACIÓN Y CARGA DE DATOS GLOBAL ---
FILE_PATH = "../DATA/PROCESSED/PROCESSED_GL_DATA_ALL_P_L_ANNUAL_MONTHLY_PER_ENTERPRISEV6.csv"
LOOK_BACK = 12 # Ventana de tiempo: Usar 12 meses para predecir el siguiente.
EPOCHS = 1000
LEARNING_RATE = 0.0005
FUTURE_MONTHS = 3
MIN_DATA_POINTS = LOOK_BACK * 2 + 1
N_SPLITS = 5 # CONSTANTE PARA LA VALIDACIÓN CRUZADA

OUTPUT_DIR = "../REPORTS/RESULT_LSTM"
MODELS_DIR = "../MODELS/"
# MODELS_DIR = os.path.join(OUTPUT_DIR, "MODELS_PORTFOLIO_LSTM")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# ----------------------------------------------------
# --- CARGA Y FILTRADO INICIAL ---
# ----------------------------------------------------

try:
    df_full = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo {FILE_PATH}.")
    print("Por favor, asegúrate de que el archivo esté en la ruta correcta.")
    exit()

# Filtrado por PORTFOLIOID para el grupo
ENTERPRISE_ID_TO_PROCESS = 0
df_full = df_full[df_full['PORTFOLIOID'] > ENTERPRISE_ID_TO_PROCESS].copy()

print(f"✅ Datos filtrados: Solo se consideran PORTFOLIOID > {ENTERPRISE_ID_TO_PROCESS} para el análisis grupal.")
print(f"Número de registros después del filtrado por ID de PORTFOLIOID: {len(df_full)}")

✅ Datos filtrados: Solo se consideran PORTFOLIOID > 0 para el análisis grupal.
Número de registros después del filtrado por ID de PORTFOLIOID: 177


In [100]:
logging.info("2. PREPROCESAMIENTO GLOBAL")
# --- 2. PREPROCESAMIENTO GLOBAL ---

df_full['NETINCOME'] = pd.to_numeric(df_full['NETINCOME'], errors='coerce')
df_full = df_full[(df_full['MONTHID'] >= 1) & (df_full['MONTHID'] <= 12)].copy()
df_full.dropna(subset=['NETINCOME', 'YEARID', 'MONTHID', 'PORTFOLIOID', 'PORTFOLIONAME'], inplace=True)
df_full['DATE'] = pd.to_datetime(df_full['YEARID'].astype(str) + '-' + df_full['MONTHID'].astype(str) + '-01')

portfolio_ids = df_full['PORTFOLIOID'].unique()
if len(portfolio_ids) == 0:
    print("ERROR: El DataFrame no contiene PORTFOLIOID válidos después de la limpieza y el filtrado.")
    exit()

print(f"Total de portafolios a analizar: {len(portfolio_ids)}")
print("-" * 50)

# --- ESTRUCTURA GLOBAL PARA ALMACENAR RESULTADOS ---
all_portfolio_results = []
# ----------------------------------------------------


Total de portafolios a analizar: 4
--------------------------------------------------


In [101]:
logging.info("3. FUNCIONES AUXILIARES")
# --- 3. FUNCIONES AUXILIARES ---

class EpochLogger(Callback):
    """Callback personalizado para mostrar el inicio y el final de cada época."""
    def on_epoch_begin(self, epoch, logs=None):
        total_epochs = self.params.get('epochs', EPOCHS)
        if (epoch + 1) % 10 == 1 or epoch == 0:
            print(f"\n[🚀] INICIO DE LA ÉPOCA {epoch + 1}/{total_epochs}")

    def on_epoch_end(self, epoch, logs=None):
        loss = logs.get('loss', 'N/A')
        total_epochs = self.params.get('epochs', EPOCHS)
        if (epoch + 1) % 10 == 0 or (epoch + 1) == total_epochs:
            print(f"[✅] FIN DE LA ÉPOCA {epoch + 1}/{total_epochs} | Pérdida (Loss): {loss:.6f}")


def create_dataset(dataset, look_back=1):
    """Crea las secuencias de entrada (X) y salida (Y) para el modelo LSTM."""
    X, Y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), 0])
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)


def build_model(look_back, learning_rate):
    """Define y compila el modelo LSTM."""
    model = Sequential([
        Input(shape=(look_back, 1)), 
        LSTM(256, return_sequences=True), 
        Dropout(0.3),
        LSTM(128),
        Dropout(0.3),
        Dense(1)
    ])
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model


def calculate_aggregated_metrics(data_full_total, portfolio_name):
    """
    Calcula el RMSE, MAE y R2 del portafolio agregado mediante un split 80/20.
    """
    data_len = len(data_full_total)
    
    if data_len < MIN_DATA_POINTS:
        return np.nan, np.nan, np.nan

    # 1. Escalado
    scaler_agg = StandardScaler()
    scaled_data_full = scaler_agg.fit_transform(data_full_total)

    # 2. División (80% Entrenamiento, 20% Prueba)
    train_size = int(data_len * 0.80)
    train_data = scaled_data_full[0:train_size, :]
    test_data = scaled_data_full[train_size - LOOK_BACK:, :]

    # 3. Creación de Dataset
    X_train, y_train = create_dataset(train_data, LOOK_BACK)
    X_test, y_test = create_dataset(test_data, LOOK_BACK)

    if len(X_test) == 0:
        return np.nan, np.nan, np.nan

    # 4. Entrenamiento del Modelo Agregado
    model_agg = build_model(LOOK_BACK, LEARNING_RATE)
    X_train_reshaped = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
    
    model_agg.fit(
        X_train_reshaped,
        y_train,
        epochs=EPOCHS,
        batch_size=1,
        verbose=0,
        callbacks=[EarlyStopping(monitor='loss', patience=3, verbose=0, restore_best_weights=True)]
    )

    # 5. Predicción en el set de prueba
    X_test_reshaped = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
    test_predict_scaled = model_agg.predict(X_test_reshaped, verbose=0)

    # 6. Invertir escalado
    test_predict = scaler_agg.inverse_transform(test_predict_scaled)
    y_test_original = scaler_agg.inverse_transform(y_test.reshape(-1, 1))

    # 7. Cálculo de Métricas
    rmse = np.sqrt(mean_squared_error(y_test_original, test_predict))
    mae = mean_absolute_error(y_test_original, test_predict)
    r2 = r2_score(y_test_original, test_predict)
    
    return rmse, mae, r2


# --------------------------------------------------------------------------------------
# --- FUNCIÓN DE PROCESAMIENTO DE ENTIDAD CON CV ---
# --------------------------------------------------------------------------------------

def process_entity_for_portfolio(entity_id, portfolio_id, portfolio_name, df_full):
    """
    Procesa, entrena el modelo usando CV, pronostica y GUARDA el modelo para una ENTIDAD específica.
    """
    # 3.1 FILTRADO Y VERIFICACIÓN
    df_entity = df_full[
        (df_full['PORTFOLIOID'] == entity_id)  
    ].sort_values('DATE').reset_index(drop=True)

    if df_entity.empty:
        print(f"ADVERTENCIA: Entidad ID {entity_id} del Portafolio {portfolio_id} sin datos. Saltando.")
        return None, None, None, None, None 

    data_full = df_entity['NETINCOME'].values.reshape(-1, 1)
    
    print(f"\n--- 📈 Procesando Entidad: ID {entity_id} (Portafolio {portfolio_name}) (Total de datos: {len(data_full)}) ---")

    if len(data_full) < MIN_DATA_POINTS:
        print(f"ADVERTENCIA: Entidad ID {entity_id} necesita al menos {MIN_DATA_POINTS} registros. Solo se tienen {len(data_full)}. Saltando.")
        return None, None, None, None, None

    # 3.2 ESCALADO DE DATOS
    scaler = StandardScaler()
    scaled_data_full = scaler.fit_transform(data_full)
    
    # 3.3 DIVISIÓN INICIAL PARA ENTRENAMIENTO/VALIDACIÓN (80% de los datos totales)
    data_len = len(scaled_data_full)
    train_val_size = int(data_len * 0.80)
    train_val_data = scaled_data_full[0:train_val_size, :]
    
    # Generar secuencias X, Y para el conjunto de entrenamiento/validación
    X_train_val_all, y_train_val_all = create_dataset(train_val_data, LOOK_BACK)
    
    if len(X_train_val_all) == 0:
        print(f"ADVERTENCIA: Datos insuficientes para crear dataset de entrenamiento/validación para la Entidad {entity_id}. Saltando.")
        return None, None, None, None, None

    # 3.4 VALIDACIÓN CRUZADA (K-Fold)
    print(f"--- 🧪 Iniciando {N_SPLITS}-Fold Cross-Validation para la Entidad {entity_id} ---")
    kf = KFold(n_splits=N_SPLITS, shuffle=False)
    
    rmse_scores = []
    mae_scores = []
    r2_scores = []
    
    for fold_idx, (train_index, val_index) in enumerate(kf.split(X_train_val_all)):
        
        X_train, X_val = X_train_val_all[train_index], X_train_val_all[val_index]
        y_train, y_val = y_train_val_all[train_index], y_train_val_all[val_index]

        X_train_reshaped = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
        X_val_reshaped = np.reshape(X_val, (X_val.shape[0], X_val.shape[1], 1))
        
        model_fold = build_model(LOOK_BACK, LEARNING_RATE)
        
        model_fold.fit(
            X_train_reshaped,
            y_train,
            epochs=EPOCHS,
            batch_size=1,
            verbose=0,
            callbacks=[EarlyStopping(monitor='loss', patience=3, verbose=0, restore_best_weights=True)]
        )

        val_predict_scaled = model_fold.predict(X_val_reshaped, verbose=0)
        
        val_predict_original = scaler.inverse_transform(val_predict_scaled)
        y_val_original = scaler.inverse_transform(y_val.reshape(-1, 1))

        rmse_scores.append(np.sqrt(mean_squared_error(y_val_original, val_predict_original)))
        mae_scores.append(mean_absolute_error(y_val_original, val_predict_original))
        r2_scores.append(r2_score(y_val_original, val_predict_original))

    rmse_cv_avg = np.mean(rmse_scores)
    mae_cv_avg = np.mean(mae_scores)
    r2_cv_avg = np.mean(r2_scores)
    
    print(f"✅ Validación Cruzada Completada. RMSE Promedio: {rmse_cv_avg:.4f} | MAE Promedio: {mae_cv_avg:.4f} | R2 Promedio: {r2_cv_avg:.4f}")
    
    # 3.5 ENTRENAMIENTO FINAL EN TODO EL CONJUNTO DE CV
    print("--- 🧠 Reentrenamiento final en el 80% de los datos de la entidad ---")
    
    model_final = build_model(LOOK_BACK, LEARNING_RATE)
    
    X_train_val_reshaped = np.reshape(X_train_val_all, (X_train_val_all.shape[0], X_train_val_all.shape[1], 1))
    
    model_final.fit(
        X_train_val_reshaped,
        y_train_val_all,
        epochs=EPOCHS,
        batch_size=1,
        verbose=0,
        callbacks=[EarlyStopping(monitor='loss', patience=3, verbose=0, restore_best_weights=True)]
    )
    
    # --- GUARDAR EL MODELO FINAL ---
    model_filename = os.path.join(MODELS_DIR, f"lstm_entity_{entity_id}_p{portfolio_id}.keras")
    model_final.save(model_filename)
    logging.info(f"Modelo guardado en: {model_filename}")
    print(f"[💾] Modelo guardado en: {model_filename}")
    
    # 3.6 PRONÓSTICO MULTI-PASO
    forecast = []
    last_sequence = scaled_data_full[-LOOK_BACK:]
    current_input = last_sequence.reshape(1, LOOK_BACK, 1)

    for i in range(FUTURE_MONTHS):
        next_prediction = model_final.predict(current_input, verbose=0)
        forecast.append(next_prediction[0, 0])
        new_sequence = np.append(current_input[:, 1:, :], next_prediction.reshape(1, 1, 1), axis=1)
        current_input = new_sequence

    # Invertir el escalado del pronóstico
    forecast_original = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    
    return forecast_original, len(data_full), rmse_cv_avg, mae_cv_avg, r2_cv_avg


def process_portfolio_by_entity(portfolio_id, df_full, all_portfolio_results):
    """
    Función principal para el portafolio:
    1. Acumula el pronóstico de las entidades.
    2. Calcula el NETINCOME REAL total del portafolio.
    3. Calcula las métricas del portafolio agregado (RMSE, MAE, R2).
    """
    
    # 1. FILTRO DEL PORTAFOLIO Y DATOS
    df_portfolio_group = df_full[df_full['PORTFOLIOID'] == portfolio_id].copy()
    if df_portfolio_group.empty: return

    portfolio_name = df_portfolio_group['PORTFOLIONAME'].iloc[0]
    
    # 2. IDENTIFICAR LAS ENTIDADES INDIVIDUALES A MODELAR (Usamos PORTFOLIOID)
    entity_ids = df_portfolio_group['PORTFOLIOID'].unique()
    
    print("\n" + "#" * 80)
    print(f"### 🎯 INICIANDO ANÁLISIS ACUMULADO PARA PORTAFOLIO: ID {portfolio_id} - {portfolio_name}")
    print(f"### Entidades a procesar: {len(entity_ids)}")
    print("#" * 80)

    # 3. ACUMULACIÓN DE PRONÓSTICOS Y MÉTRICAS CV
    total_forecast = np.zeros(FUTURE_MONTHS)
    successful_entities = 0
    # Listas para acumular las métricas CV de las entidades
    all_rmse_cv = []
    all_mae_cv = []
    all_r2_cv = []

    for e_id in entity_ids:
        # Recibe 3 métricas CV
        entity_forecast, data_points, rmse_cv, mae_cv, r2_cv = process_entity_for_portfolio(e_id, portfolio_id, portfolio_name, df_full)
        
        if entity_forecast is not None:
            total_forecast += entity_forecast
            successful_entities += 1
            all_rmse_cv.append(rmse_cv)
            all_mae_cv.append(mae_cv)
            all_r2_cv.append(r2_cv)
        
    if successful_entities == 0:
        print(f"⚠️ Portafolio {portfolio_id} saltado: Ninguna entidad cumplió con los requisitos mínimos de datos.")
        return

    # 4. CÁLCULO DEL NETINCOME REAL TOTAL DEL PORTAFOLIO
    df_portfolio_total = df_portfolio_group.groupby('DATE')['NETINCOME'].sum().reset_index()
    data_full_total = df_portfolio_total['NETINCOME'].values.reshape(-1, 1)
    
    # 5. CÁLCULO DE MÉTRICAS DEL PORTAFOLIO AGREGADO (RMSE, MAE, R2)
    print("\n--- 📊 Calculando Métricas para la serie de tiempo TOTAL del Portafolio ---")
    rmse_agg, mae_agg, r2_agg = calculate_aggregated_metrics(data_full_total, portfolio_name)
    
    # 6. Almacenar resultados
    
    # Promedio de las métricas CV de todas las entidades
    rmse_cv_avg_report = np.mean(all_rmse_cv) if all_rmse_cv else np.nan
    mae_cv_avg_report = np.mean(all_mae_cv) if all_mae_cv else np.nan
    r2_cv_avg_report = np.mean(all_r2_cv) if all_r2_cv else np.nan

    all_portfolio_results.append({
        'PORTFOLIOID': portfolio_id,
        'PORTFOLIONAME': portfolio_name,
        'RMSE_CV_AVG': rmse_cv_avg_report,
        'MAE_CV_AVG': mae_cv_avg_report,
        'R2_CV_AVG': r2_cv_avg_report,
        'RMSE_TEST_AGG': rmse_agg,
        'MAE_TEST_AGG': mae_agg,
        'R2_TEST_AGG': r2_agg,
        'DATA_POINTS': len(data_full_total), 
        'FORECAST_MONTHS': FUTURE_MONTHS,
        'FORECAST': total_forecast
    })
    
    print(f"Portafolio {portfolio_id} - {portfolio_name} | NETINCOME Proyectado:")
    for i, value in enumerate(total_forecast):
        print(f"Mes {i+1} (Acumulado): {value:,.2f}")


    # 7. GRÁFICO 
    plt.figure(figsize=(18, 8))
    time_axis = df_portfolio_total['DATE']
    plt.plot(time_axis, data_full_total, label='NETINCOME Total Real del Portafolio', color='blue', linewidth=2)
    last_date = time_axis.iloc[-1]
    future_dates = pd.date_range(start=last_date, periods=FUTURE_MONTHS + 1, freq='MS')[1:]
    plt.plot(future_dates, total_forecast, label=f'Pronóstico ACUMULADO ({successful_entities} Entidades)', color='red', linestyle='-', marker='o', linewidth=2)
    plt.title(f'Pronóstico ACUMULADO de NETINCOME para el Portafolio: {portfolio_name} (ID: {portfolio_id})')
    plt.xlabel('Fecha')
    plt.ylabel('NETINCOME Acumulado')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plot_save_path = os.path.join(OUTPUT_DIR, f"forecast_portfolio_{portfolio_id}_accumulated.png")
    plt.savefig(plot_save_path)
    plt.close()
    print(f"Gráfico ACUMULADO guardado en: {plot_save_path}")
    print("-" * 50)


In [102]:
logging.info("4. EJECUCIÓN DEL ANÁLISIS PARA TODOS LOS PORTAFOLIOS")
# --- 4. EJECUCIÓN DEL ANÁLISIS PARA TODOS LOS PORTAFOLIOS ---

for p_id in portfolio_ids:
    process_portfolio_by_entity(p_id, df_full, all_portfolio_results)

print("\n🎉 Análisis, pronóstico y ACUMULACIÓN completado. Revisa la carpeta './REPORTS'.")

logging.info("5. GENERACIÓN DEL RESUMEN FINAL Y EXPORTACIÓN A EXCEL (NUEVA IMPLEMENTACIÓN)")
# ------------------------------------------------------------------------------------------
# --- 5. GENERACIÓN DEL RESUMEN FINAL Y EXPORTACIÓN A EXCEL (NUEVA IMPLEMENTACIÓN) ---
# ------------------------------------------------------------------------------------------

if all_portfolio_results:
    # 5.1 PREPARAR DATOS PARA EL DATAFRAME
    summary_data_list = []
    forecast_headers = [f'Pronóst.Mes {i+1}' for i in range(FUTURE_MONTHS)]

    for result in all_portfolio_results:
        row = {
            'Portafolio': result['PORTFOLIONAME'],
            '# Puntos': result['DATA_POINTS'],
            'MAE(CV)': result['MAE_CV_AVG'],
            'RMSE(CV)': result['RMSE_CV_AVG'],
            'MAE(Agg P)': result['MAE_TEST_AGG'],
            'RMSE(Agg P)': result['RMSE_TEST_AGG'],
        }
        # Agregar los pronósticos
        for i, val in enumerate(result['FORECAST']):
            row[forecast_headers[i]] = val
        
        summary_data_list.append(row)

    df_summary = pd.DataFrame(summary_data_list)
    
    # 5.2 EXPORTAR A EXCEL {EPOCHS}
    excel_filename = os.path.join(OUTPUT_DIR, f"SUMMARY_RESULTS_PORTFOLIO_LSTM_EPOCH{EPOCHS}.xlsx")
    logging.info("Genera Excel con el resultado de las metricas")
    
    
    try:
        df_summary.to_excel(excel_filename, index=False, sheet_name='REPORT')
        print(f"\n[📄] **Archivo Excel con el resumen guardado exitosamente en:** {excel_filename}")
    except Exception as e:
        print(f"\n⚠️ ERROR al intentar guardar el archivo Excel: {e}")
        print("Asegúrese de tener instalada la librería 'openpyxl' (pip install openpyxl) si usa un entorno virtual.")


    # 5.3 IMPRIMIR RESUMEN EN CONSOLA (TABULATE)
    
    last_real_date = df_full['DATE'].max().strftime('%Y-%m')
    
    headers_metrics = [
        "MAE (CV)", "RMSE (CV)", 
        "MAE (Agg P)", "RMSE (Agg P)" 
    ]

    headers = ["Portafolio", "# Puntos"] + headers_metrics + forecast_headers
    
    # Preparamos los datos de la tabla para impresión (formateo con comas y decimales)
    tabulate_data = []
    for result in all_portfolio_results:
        mae_cv_val = f"{result['MAE_CV_AVG']:,.2f}" if not np.isnan(result['MAE_CV_AVG']) else "N/A"
        rmse_cv_val = f"{result['RMSE_CV_AVG']:,.2f}" if not np.isnan(result['RMSE_CV_AVG']) else "N/A"
        mae_agg_val = f"{result['MAE_TEST_AGG']:,.2f}" if not np.isnan(result['MAE_TEST_AGG']) else "Insuficiente"
        rmse_agg_val = f"{result['RMSE_TEST_AGG']:,.2f}" if not np.isnan(result['RMSE_TEST_AGG']) else "Insuficiente"
        row = [
            result['PORTFOLIONAME'],            
            f"{result['DATA_POINTS']}",
            mae_cv_val, rmse_cv_val, 
            mae_agg_val, rmse_agg_val, 
        ]
        row.extend([f"{val:,.2f}" for val in result['FORECAST']])
        tabulate_data.append(row)

    print("\n" + "="*160)
    print("                              RESUMEN DE RESULTADOS DE PRONÓSTICO LSTM (NETINCOME) ACUMULADO POR PORTAFOLIO")
    print("="*160)
    print(f"  Modelo: LSTM (Lookback={LOOK_BACK}, Épocas Máx={EPOCHS}, LR={LEARNING_RATE})")
    print(f"  Metodología: CV (5-Fold) en entidad individual. Pronóstico sumado. Métricas de prueba calculadas en serie de tiempo agregada.")
    print(f"  Pronóstico a {FUTURE_MONTHS} meses a partir de la última fecha real disponible (~{last_real_date}).")
    print("-" * 160)

    print(tabulate(tabulate_data, headers=headers, tablefmt="fancy_grid"))

    print("\n## Interpretación del Resumen de Métricas")
    print("### Métricas CV (Entidad Avg)")
    print("Estas métricas representan el **promedio** de los errores de validación cruzada ($5$-Fold) a nivel de la entidad individual. Indican la estabilidad y calidad del modelo entrenado para cada componente del portafolio.")
    print(f" - **MAE (CV):** Error absoluto promedio en unidades monetarias. Menor es mejor.")
    print(f" - **RMSE (CV):** Error cuadrático promedio, penalizando más los errores grandes. Menor es mejor.")
    print(f" - **$R^2$ (CV):** Proporción de la varianza explicada por el modelo. Más cercano a $1.0$ es mejor.")
    print("\n### Métricas Prueba (Agg)")
    print("Estas métricas se calculan sobre el $20%$ final de la **serie de tiempo TOTAL (agregada)** del portafolio, utilizando un modelo entrenado específicamente en esa serie. Son la referencia de la precisión del portafolio como un todo.")
    print(f" - **MAE (Agg P):** Error absoluto promedio en unidades monetarias del pronóstico del agregado. Menor es mejor.")
    print(f" - **RMSE (Agg P):** Error cuadrático promedio del pronóstico del agregado. Menor es mejor.")
    print(f" - **$R^2$ (Agg P):** Proporción de la varianza explicada por el modelo del agregado. Más cercano a $1.0$ es mejor.")
    logging.info("FINALIZA ENTRENAMIENTO")

else:
    print("\n⚠️ No se pudieron generar resultados para el resumen final. Verifica las advertencias de los portafolios que se saltaron.")


################################################################################
### 🎯 INICIANDO ANÁLISIS ACUMULADO PARA PORTAFOLIO: ID 1.0 - L2-ALL
### Entidades a procesar: 1
################################################################################

--- 📈 Procesando Entidad: ID 1.0 (Portafolio L2-ALL) (Total de datos: 60) ---
--- 🧪 Iniciando 5-Fold Cross-Validation para la Entidad 1.0 ---
✅ Validación Cruzada Completada. RMSE Promedio: 2863400.7016 | MAE Promedio: 2028702.7772 | R2 Promedio: -12.5902
--- 🧠 Reentrenamiento final en el 80% de los datos de la entidad ---
[💾] Modelo guardado en: ../MODELS/lstm_entity_1.0_p1.0.keras

--- 📊 Calculando Métricas para la serie de tiempo TOTAL del Portafolio ---
Portafolio 1.0 - L2-ALL | NETINCOME Proyectado:
Mes 1 (Acumulado): 1,557,727.25
Mes 2 (Acumulado): 1,507,287.88
Mes 3 (Acumulado): 1,562,874.00
Gráfico ACUMULADO guardado en: ../REPORTS/RESULT_LSTM\forecast_portfolio_1.0_accumulated.png
-----------------------------------------